# LLM Training as Large-scale Optimization

## Stochastic Optimization settings
In this chapter we will discuss the training of LLMs as a large-scale optimization problem.

We have already discussed stochastic optimization problem:
```{math}
:label: eq-stoc-opt-chap4
\min_{\vw}\ f(\vw) := \EE_{\xi\sim\mathcal{D}}[\ell(\vw,\xi)]
```
where $\mathcal{D}$ is the data distribution. Note that for neural network training, we usually collect a set of training data $\{\xi_i\}_{i=1,2,...,n}$ and optimize for the training loss
```{math}
:label: eq-stoc-opt-finite-sum-chap4
\min_{\vw}\ f(\vw) := \frac{1}{n}\sum_{i=1}^{n}\ell(\vw, \xi_i)
```
which is also know as the finite sum setting of the stochastic optimization.

## SGD and its convergence

The SGD method aims to tackle {eq}`eq-stoc-opt-chap4` by mimicking the gradient descent method applied on the stochastic optimization problem. In particular, observe that if we have $\xi \sim {\cal D}$, then 
```{math}
\nabla f( \vw ) \approx \nabla \ell( \vw, \xi ) 
```
The above insight motivates the SGD method as follows:

````{prf:algorithm} Stochastic Gradient (SGD) Method
:label: alg:SGD

- **Input**: initial point $\vw^0$, stepsize sequence $\{ \gamma_k \}_{k \geq 0}$, max. no. of iterations $K$.

- For $k=0,1,2,..., K-1$,
```{math}
\begin{split}
& \text{draw sample}~\xi^{k+1} \sim {\cal D} \\
& \text{update}~\vw^{k+1} = \vw^k - \gamma_k \nabla \ell( \vw^k, \xi^{k+1} ).
\end{split}
```
- **Output**: last iterate $\vw^K$, or the solution sequence $\{ \vw^k \}_{k=1}^K$.
````

Unlike the deterministic gradient method, the sequence $\{ \vw^k \}_{k=1}^K$ generated by the SGD method is a **random process** itself (in fact, a Markov chain). As such, there are several subtleties in analyzing the convergence properties of the solution sequence. Here, the common forms of convergence are *almost-sure convergence*, *convergence in expectation*, etc. We will demonstrate the **convergence in expectation** of SGD under the simplest setting, i.e., with smooth (but possibly non-convex) objective function. Interested readers may refer to (ref.) for further discussions. 

In addition to {prf:ref}`assm:smooth` on the expected objective function $f(\vw)$, we need the following condition on the stochastic gradient: 

````{prf:assumption} Stochastic Oracle
:label: assm:stochastic

There exists $\sigma \geq 0$ such that for any $\vw \in \mathbb{R}^d$, we have 
```{math}
\begin{split}
& \EE_{ \xi \sim {\cal D} } [ \nabla \ell( \vw, \xi ) ] = \nabla f( \vw ) \\
& \EE_{ \xi \sim {\cal D} } [ \| \nabla \ell( \vw, \xi ) - \nabla f( \vw ) \|^2 ] \leq \sigma^2
\end{split}
```
````
In other words, the stochastic oracle used in the SGD method is both **unbiased** and has **bounded variance**. 

We observe the following *convergence in expecatation* result: 

````{prf:theorem} Convergence of SGD Method (Smooth Case)
:label: thm:sgd-smooth 

Under {prf:ref}`assm:smooth`, {prf:ref}`assm:stochastic` and suppose that the step size satisfies $\gamma_k \leq 1/L$. For any $K \geq 1$, the following holds:

```{math}
\min_{ k=0, ..., K-1 } \EE[ \| \nabla f( \vw^k ) \|^2 ] \leq \frac{ 2 \left( f (\vw^0) - f(\vw^K) + \frac{ \sigma^2 L }{2} \sum_{k=0}^{K-1} \gamma_k^2 \right) }{ \sum_{k=0}^{K-1} \gamma_k }
```

````

````{admonition} Proof (SGD for Smooth Objective Function)
:class: dropdown

The proof is very similar to that of {prf:ref}`thm:gd-smooth`. Observing that under {prf:ref}`assm:smooth`, it holds for any $k \geq 0$ that, 
```{math}
f( \vw^{k+1} ) \leq f( \vw^k ) + \langle \nabla f( \vw^k ) , \vw^{k+1} - \vw^k \rangle + \frac{L}{2} \| \vw^{k+1} - \vw^k \|^2
```

By noting that $\vw^{k+1} - \vw^k = - \gamma_k \nabla \ell (\vw^k; \xi^{k+1})$, we have
```{math}
:label: eq-sgd-analyze-step1
f( \vw^{k+1} ) \leq f( \vw^k ) - \gamma_k \langle \nabla f( \vw^k ), \nabla \ell (\vw^k; \xi^{k+1}) \rangle + \frac{ \gamma_k^2 L }{2} \| \nabla \ell (\vw^k; \xi^{k+1}) \|^2 
```

We denote ${\cal F}_k$ as the filtration of the random variables $\{ \vw^0, \xi^1, \ldots, \xi^k, \vw^k \}$. 
Under {prf:ref}`assm:stochastic`, we have the following conditional expectations:
```{math}
\EE[ \langle \nabla f( \vw^k ), \nabla \ell (\vw^k; \xi^{k+1}) \rangle | {\cal F}_k ] = \| \nabla f( \vw^k ) \|^2
```
and 
```{math}
\begin{split}
& \EE[ \| \nabla \ell (\vw^k; \xi^{k+1}) \|^2 | {\cal F}_k ] \\
& = \EE[ \| \nabla \ell (\vw^k; \xi^{k+1}) - \nabla f( \vw^k ) \|^2 | {\cal F}_k ] + \| \nabla f( \vw^k ) \|^2 \\
& \leq \sigma^2 + \| \nabla f( \vw^k ) \|^2.
\end{split}
```
Taking the full expectation on both sides of {eq}`eq-sgd-analyze-step1` yields
```{math}
\EE[ f( \vw^{k+1} ) ] \leq \EE[ f(\vw^k) ] - \gamma_k \left( 1 - \frac{\gamma_k L}{2} \right) \EE[ \| \nabla f( \vw^k ) \|^2 ] + \frac{ \gamma_k^2 \sigma^2 L }{2}
```
Using $\gamma_k \leq 1/L$ simplifies the above to
```{math}
\EE[ f( \vw^{k+1} ) ] \leq \EE[ f(\vw^k) ] - \frac{\gamma_k}{2} \EE[ \| \nabla f( \vw^k ) \|^2 ] + \frac{ \gamma_k^2 \sigma^2 L }{2}
```

This implies 
```{math}
\frac{\gamma_k}{2} \EE[ \| \nabla f( \vw^k ) \|^2 ] \leq \EE[ f( \vw^k ) - f( \vw^{k+1} ) ] + \frac{ \gamma_k^2 \sigma^2 L }{2}.
```
Summing both sides from $k=0$ to $k=K-1$ yields the desired result.

````

From the results in {prf:ref}`thm:sgd-smooth`, we notice that if we set $\gamma_k = 1 / \sqrt{K}$ for all $k=0,...,K-1$, then one has
```{math}
\min_{ k=0, ..., K-1 } \EE[ \| \nabla f( \vw^k ) \|^2 ] = {\cal O} \left( \frac{1 + \log K}{ \sqrt{K} } \right),
```
showing the convergence (in expectation) for the SGD method.
